In [1]:
import pandas as pd
import vivarium_inputs
import vivarium.gbd_mapping as gbd_mapping
import pathlib
from lsff_utils import config_utils
from lsff_utils.results import expand_to_all_scenarios, aggregate_by_scenario

In [2]:
location = "india"
vehicle = "rice"

In [3]:
# Parameters
location = "nigeria"
vehicle = "bouillon"


In [4]:
scenarios = list(
    config_utils.get_location_fortificant_vehicle_intervention_scenarios()
    .pipe(lambda df: df[(df.location == location) & (df.vehicle == vehicle)])
    .intervention_scenario.unique()
) + ["zero", "baseline"]
scenarios

['intervention', 'zero', 'baseline']

In [5]:
path = f"results/rescaled_pregnancy_results/{vehicle}/{location}/person_time_anemia.parquet"
if pathlib.Path(path).is_file():
    pregnancy_person_time_anemia = pd.read_parquet(path)
else:
    pregnancy_person_time_anemia = expand_to_all_scenarios(
        pd.read_parquet(
            f"results/rescaled_pregnancy_results/rice/india/person_time_anemia.parquet"
        ).assign(value=0),
        scenarios,
    )
pregnancy_person_time_anemia

,measure,entity_type,entity,sub_entity,age_group,anemia_status_at_birth,wealth_quintile,scenario,input_draw,random_seed,value
0,person_time,impairment,anemia,not_anemic,10_to_14,invalid,1,baseline,0,2,702.717225
1,person_time,impairment,anemia,not_anemic,10_to_14,invalid,2,baseline,0,2,700.223109
2,person_time,impairment,anemia,not_anemic,10_to_14,invalid,3,baseline,0,2,653.458431
3,person_time,impairment,anemia,not_anemic,10_to_14,invalid,4,baseline,0,2,641.611379
4,person_time,impairment,anemia,not_anemic,10_to_14,invalid,5,baseline,0,2,924.070033
...,...,...,...,...,...,...,...,...,...,...,...
53995,person_time,impairment,anemia,severe,95_plus,severe,1,baseline,0,9,0.000000
53996,person_time,impairment,anemia,severe,95_plus,severe,2,baseline,0,9,0.000000
53997,person_time,impairment,anemia,severe,95_plus,severe,3,baseline,0,9,0.000000
53998,person_time,impairment,anemia,severe,95_plus,severe,4,baseline,0,9,0.000000


In [6]:
pregnancy_person_time_anemia.groupby("scenario").random_seed.nunique()

scenario
baseline        10
intervention    10
zero            10
Name: random_seed, dtype: int64

In [7]:
pregnancy_person_time_anemia.sub_entity.value_counts()

not_anemic    13500
mild          13500
moderate      13500
severe        13500
Name: sub_entity, dtype: int64

In [8]:
total_pregnant_person_time = aggregate_by_scenario(pregnancy_person_time_anemia)
total_pregnant_person_time

scenario      wealth_quintile
baseline      1                  2.480785e+06
              2                  2.583073e+06
              3                  2.247129e+06
              4                  2.015605e+06
              5                  1.530577e+06
intervention  1                  2.480833e+06
              2                  2.583091e+06
              3                  2.247151e+06
              4                  2.015653e+06
              5                  1.530604e+06
zero          1                  2.480785e+06
              2                  2.583073e+06
              3                  2.247129e+06
              4                  2.015605e+06
              5                  1.530577e+06
Name: value, dtype: float64

In [9]:
anemic_pregnant_person_time = aggregate_by_scenario(
    pregnancy_person_time_anemia[
        pregnancy_person_time_anemia.sub_entity != "not_anemic"
    ]
)
anemic_pregnant_person_time

scenario      wealth_quintile
baseline      1                  1.311325e+06
              2                  1.335194e+06
              3                  1.121397e+06
              4                  1.025569e+06
              5                  7.001227e+05
intervention  1                  1.200419e+06
              2                  1.220351e+06
              3                  1.011956e+06
              4                  9.399388e+05
              5                  6.155466e+05
zero          1                  1.311325e+06
              2                  1.335194e+06
              3                  1.121397e+06
              4                  1.025569e+06
              5                  7.001227e+05
Name: value, dtype: float64

In [10]:
pregnant_anemia_prevalence_by_scenario = (
    anemic_pregnant_person_time / total_pregnant_person_time
).fillna(0)
pregnant_anemia_prevalence_by_scenario

scenario      wealth_quintile
baseline      1                  0.528593
              2                  0.516901
              3                  0.499035
              4                  0.508815
              5                  0.457424
intervention  1                  0.483877
              2                  0.472438
              3                  0.450329
              4                  0.466320
              5                  0.402159
zero          1                  0.528593
              2                  0.516901
              3                  0.499035
              4                  0.508815
              5                  0.457424
Name: value, dtype: float64

In [11]:
path = f"./results/{location}/{vehicle}/pregnant_anemia_prevalence_by_scenario.csv"
pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
pregnant_anemia_prevalence_by_scenario.to_csv(path)

In [12]:
pop = pd.read_csv(f"../0100_data_prep/results/population/stratified/{location}.csv")
pop

,sex,age_start,age_end,pregnant,wealth_quintile,value
0,Female,0.0,0.019178,not_pregnant,1,18793.149621
1,Female,0.0,0.019178,not_pregnant,2,17830.620200
2,Female,0.0,0.019178,not_pregnant,3,16129.662368
3,Female,0.0,0.019178,not_pregnant,4,14114.028154
4,Female,0.0,0.019178,not_pregnant,5,12208.445301
...,...,...,...,...,...,...
280,Male,95.0,125.000000,not_pregnant,1,4849.649774
281,Male,95.0,125.000000,not_pregnant,2,4371.181536
282,Male,95.0,125.000000,not_pregnant,3,4569.918352
283,Male,95.0,125.000000,not_pregnant,4,4550.906874


In [13]:
pregnant_pop = pop[pop.pregnant == "pregnant"].groupby(["wealth_quintile"]).value.sum()
pregnant_pop

wealth_quintile
1    2.306716e+06
2    2.394711e+06
3    2.088829e+06
4    1.862288e+06
5    1.411894e+06
Name: value, dtype: float64

In [14]:
pregnancy_prevalent_anemia_cases_by_scenario = (
    pregnant_anemia_prevalence_by_scenario * pregnant_pop
)
pregnancy_prevalent_anemia_cases_by_scenario

scenario      wealth_quintile
baseline      1                  1.219314e+06
              2                  1.237829e+06
              3                  1.042399e+06
              4                  9.475592e+05
              5                  6.458342e+05
intervention  1                  1.116168e+06
              2                  1.131353e+06
              3                  9.406592e+05
              4                  8.684216e+05
              5                  5.678066e+05
zero          1                  1.219314e+06
              2                  1.237829e+06
              3                  1.042399e+06
              4                  9.475592e+05
              5                  6.458342e+05
Name: value, dtype: float64

In [15]:
path = f"results/rescaled_pregnancy_results/{vehicle}/{location}/transition_count_maternal_disorders.parquet"
if pathlib.Path(path).is_file():
    maternal_disorders_transition_counts = pd.read_parquet(path)
else:
    maternal_disorders_transition_counts = expand_to_all_scenarios(
        pd.read_parquet(
            f"results/rescaled_pregnancy_results/rice/india/transition_count_maternal_disorders.parquet"
        ).assign(value=0),
        scenarios,
    )

maternal_disorders_transition_counts

,measure,entity_type,entity,sub_entity,age_group,anemia_status_at_birth,wealth_quintile,scenario,input_draw,random_seed,value
0,transition_count,cause,maternal_disorders,susceptible_to_maternal_disorders_to_maternal_...,10_to_14,invalid,1,baseline,0,2,0.0
1,transition_count,cause,maternal_disorders,susceptible_to_maternal_disorders_to_maternal_...,10_to_14,invalid,2,baseline,0,2,0.0
2,transition_count,cause,maternal_disorders,susceptible_to_maternal_disorders_to_maternal_...,10_to_14,invalid,3,baseline,0,2,0.0
3,transition_count,cause,maternal_disorders,susceptible_to_maternal_disorders_to_maternal_...,10_to_14,invalid,4,baseline,0,2,0.0
4,transition_count,cause,maternal_disorders,susceptible_to_maternal_disorders_to_maternal_...,10_to_14,invalid,5,baseline,0,2,0.0
...,...,...,...,...,...,...,...,...,...,...,...
26995,transition_count,cause,maternal_disorders,maternal_disorders_to_recovered_from_maternal_...,95_plus,severe,1,baseline,0,9,0.0
26996,transition_count,cause,maternal_disorders,maternal_disorders_to_recovered_from_maternal_...,95_plus,severe,2,baseline,0,9,0.0
26997,transition_count,cause,maternal_disorders,maternal_disorders_to_recovered_from_maternal_...,95_plus,severe,3,baseline,0,9,0.0
26998,transition_count,cause,maternal_disorders,maternal_disorders_to_recovered_from_maternal_...,95_plus,severe,4,baseline,0,9,0.0


In [16]:
maternal_disorders_transition_counts.sub_entity.cat.categories

Index(['susceptible_to_maternal_disorders_to_maternal_disorders', 'maternal_disorders_to_recovered_from_maternal_disorders'], dtype='object')

In [17]:
maternal_disorders_incident_cases_by_scenario = aggregate_by_scenario(
    maternal_disorders_transition_counts[
        maternal_disorders_transition_counts.sub_entity
        == "susceptible_to_maternal_disorders_to_maternal_disorders"
    ]
)
maternal_disorders_incident_cases_by_scenario

scenario      wealth_quintile
baseline      1                  2.561860e+06
              2                  2.224473e+06
              3                  2.110146e+06
              4                  1.934522e+06
              5                  1.414713e+06
intervention  1                  2.494610e+06
              2                  2.157354e+06
              3                  2.045727e+06
              4                  1.881718e+06
              5                  1.372938e+06
zero          1                  2.561860e+06
              2                  2.224473e+06
              3                  2.110146e+06
              4                  1.934522e+06
              5                  1.414713e+06
Name: value, dtype: float64

In [18]:
path = (
    f"./results/{location}/{vehicle}/maternal_disorders_incident_cases_by_scenario.csv"
)
pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
maternal_disorders_incident_cases_by_scenario.to_csv(path)

In [19]:
path = f"results/rescaled_child_results/{vehicle}/{location}/deaths.parquet"

# NOTE: The child_scenario column currently contains only 'baseline'
# because we didn't have any interventions in the child simulation. If
# we add a child intervention that creates another scenario in this
# column, then results from different child scenarios would get added
# together in the call to aggregate_by_scenario below, so we'd need to
# change the processing code in that case.
def assert_unique_child_scenario(df):
    assert set(df.child_scenario.unique()) == {'baseline'}
    return df

if pathlib.Path(path).is_file():
    neonatal_deaths = (
        pd.read_parquet(path)
        .pipe(assert_unique_child_scenario)
        .rename(columns={"maternal_scenario": "scenario"})
    )
else:
    neonatal_deaths = expand_to_all_scenarios(
        pd.read_parquet(f"results/rescaled_child_results/rice/india/deaths.parquet")
        .assign(value=0)
        .rename(columns={"maternal_scenario": "scenario"}),
        scenarios,
    )

neonatal_deaths

,measure,entity_type,entity,sub_entity,age_group,sex,wealth_quintile,child_scenario,scenario,input_draw,random_seed,value
0,deaths,cause,other_causes,other_causes,0_to_5_months,Female,1,baseline,intervention,0,2,3720.335017
1,deaths,cause,other_causes,other_causes,0_to_5_months,Female,2,baseline,intervention,0,2,3720.335017
2,deaths,cause,other_causes,other_causes,0_to_5_months,Female,3,baseline,intervention,0,2,3263.451770
3,deaths,cause,other_causes,other_causes,0_to_5_months,Female,4,baseline,intervention,0,2,3002.375628
4,deaths,cause,other_causes,other_causes,0_to_5_months,Female,5,baseline,intervention,0,2,2153.878168
...,...,...,...,...,...,...,...,...,...,...,...,...
1195,deaths,cause,other_causes,other_causes,18_to_59_months,Male,1,baseline,intervention,0,6,3361.355323
1196,deaths,cause,other_causes,other_causes,18_to_59_months,Male,2,baseline,intervention,0,6,3067.644664
1197,deaths,cause,other_causes,other_causes,18_to_59_months,Male,3,baseline,intervention,0,6,3067.644664
1198,deaths,cause,other_causes,other_causes,18_to_59_months,Male,4,baseline,intervention,0,6,2251.781721


In [20]:
neonatal_deaths_by_scenario = aggregate_by_scenario(neonatal_deaths)
neonatal_deaths_by_scenario

scenario      wealth_quintile
baseline      1                  167937.228068
              2                  174496.766125
              3                  156580.415909
              4                  138598.796658
              5                  105311.588608
intervention  1                  167643.517409
              2                  174235.689983
              3                  156156.167179
              4                  138468.258588
              5                  105148.416019
zero          1                  167937.228068
              2                  174496.766125
              3                  156580.415909
              4                  138598.796658
              5                  105311.588608
Name: value, dtype: float64

In [21]:
path = f"./results/{location}/{vehicle}/neonatal_deaths_by_scenario.csv"
pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
neonatal_deaths_by_scenario.to_csv(path)

In [22]:
path = f"../0400_non_pregnant_anemia_model/results/{vehicle}/{location}/anemia_cases.parquet"
if pathlib.Path(path).is_file():
    non_pregnancy_anemia_cases = pd.read_parquet(path)
else:
    non_pregnancy_anemia_cases = expand_to_all_scenarios(
        pd.read_parquet(
            f"../0400_non_pregnant_anemia_model/results/rice/india/anemia_cases.parquet"
        ).assign(value=0),
        scenarios,
    )

non_pregnancy_anemia_cases

,sex,age_start,age_end,wealth_quintile,value,scenario
0,Female,0.0,0.019178,1,15253.787430,zero
1,Female,0.0,0.019178,2,14177.057673,zero
2,Female,0.0,0.019178,3,12677.857494,zero
3,Female,0.0,0.019178,4,10921.063707,zero
4,Female,0.0,0.019178,5,8999.215869,zero
...,...,...,...,...,...,...
745,Male,95.0,125.000000,1,3547.897967,intervention
746,Male,95.0,125.000000,2,3228.212172,intervention
747,Male,95.0,125.000000,3,3415.106422,intervention
748,Male,95.0,125.000000,4,3408.600025,intervention


In [23]:
non_pregnancy_prevalent_anemia_cases_by_scenario = aggregate_by_scenario(
    non_pregnancy_anemia_cases.assign(entity="anemia", input_draw="draw_0")
)
non_pregnancy_prevalent_anemia_cases_by_scenario

scenario      wealth_quintile
baseline      1                  2.395973e+07
              2                  2.348450e+07
              3                  2.171342e+07
              4                  2.046778e+07
              5                  1.920935e+07
intervention  1                  2.234810e+07
              2                  2.177320e+07
              3                  1.988815e+07
              4                  1.864166e+07
              5                  1.739685e+07
zero          1                  2.395973e+07
              2                  2.348450e+07
              3                  2.171342e+07
              4                  2.046778e+07
              5                  1.920935e+07
Name: value, dtype: float64

In [24]:
prevalent_anemia_cases_by_scenario = (
    pregnancy_prevalent_anemia_cases_by_scenario
    + non_pregnancy_prevalent_anemia_cases_by_scenario
)
prevalent_anemia_cases_by_scenario

scenario      wealth_quintile
baseline      1                  2.517904e+07
              2                  2.472233e+07
              3                  2.275582e+07
              4                  2.141534e+07
              5                  1.985518e+07
intervention  1                  2.346427e+07
              2                  2.290456e+07
              3                  2.082880e+07
              4                  1.951008e+07
              5                  1.796466e+07
zero          1                  2.517904e+07
              2                  2.472233e+07
              3                  2.275582e+07
              4                  2.141534e+07
              5                  1.985518e+07
Name: value, dtype: float64

In [25]:
path = f"./results/{location}/{vehicle}/prevalent_anemia_cases_by_scenario.csv"
pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
prevalent_anemia_cases_by_scenario.to_csv(path)

In [26]:
path = f"../0500_neural_tube_defects_model/results/{location}/{vehicle}/ntd_cases_by_scenario.csv"
if pathlib.Path(path).is_file():
    ntd_cases_by_scenario = pd.read_csv(path)
else:
    ntd_cases_by_scenario = expand_to_all_scenarios(
        pd.read_csv(
            f"../0500_neural_tube_defects_model/results/india/rice/ntd_cases_by_scenario.csv"
        ).assign(value=0),
        scenarios,
    )

ntd_cases_by_scenario = ntd_cases_by_scenario.set_index(
    ["scenario", "wealth_quintile"]
).value
ntd_cases_by_scenario

scenario      wealth_quintile
zero          1                  12734.955657
              2                  13022.284791
              3                  11788.751094
              4                  10449.533117
              5                   9178.019131
baseline      1                  12734.955657
              2                  13022.284791
              3                  11788.751094
              4                  10449.533117
              5                   9178.019131
intervention  1                   7997.730204
              2                   8407.265668
              3                   8326.999594
              4                   7788.892686
              5                   6991.392924
Name: value, dtype: float64

In [27]:
path = f"./results/{location}/{vehicle}/ntd_cases_by_scenario.csv"
pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
ntd_cases_by_scenario.to_csv(path)